# Évaluation du Modèle Motion Flow

Ce notebook permet de charger votre modèle entraîné sur les paysages, de prédire le champ de mouvement sur une image de test, et de visualiser le flow généré par le U-Net.

In [ ]:
import os
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from scipy.ndimage import map_coordinates

# Importer l'architecture du réseau depuis le projet
from motion_flow.model import MotionFlowUNet

## 1. Chargement du Modèle (Dernier Checkpoint)
Nous allons chercher le modèle entraîné le plus récent dans le dossier `checkpoints`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilisation de: {device}")

# Initialiser l'architecture
model = MotionFlowUNet().to(device)

# Trouver le dernier run
ckpt_dir = "../checkpoints"
if os.path.exists(ckpt_dir):
    runs = sorted(glob.glob(os.path.join(ckpt_dir, "run_*")), reverse=True)
    if runs:
        latest_run = runs[0]
        # Essayer de charger le meilleur modèle en priorité, sinon le dernier
        model_path = os.path.join(latest_run, "model_best.pth")
        if not os.path.exists(model_path):
            model_path = os.path.join(latest_run, "model_latest.pth")
            
        if os.path.exists(model_path):
            print(f"Chargement des poids depuis : {model_path}")
            model.load_state_dict(torch.load(model_path, map_location=device))
            model.eval() # Mode inférence (désactive dropout, batchnorm, etc.)
        else:
            print(f"⚠️ Aucun fichier .pth trouvé dans {latest_run}.")
            print("Le modèle utilisera donc des poids aléatoires (NON ENTRAÎNÉ).")
    else:
        print("⚠️ Aucun run trouvé dans checkpoints/.")
        print("Le modèle utilisera donc des poids aléatoires (NON ENTRAÎNÉ).")
else:
    print("⚠️ Dossier checkpoints introuvable.")
    print("Le modèle utilisera donc des poids aléatoires (NON ENTRAÎNÉ).")

## 2. Chargement de l'Image de Test
Nous choisissons une image que le modèle n'a idéalement jamais vue.

In [ ]:
# Remplacer par une image statique de votre choix
test_image_path = "../data/landscape.jpg"

if os.path.exists(test_image_path):
    pil_image = Image.open(test_image_path).convert('RGB')
    
    # Transformation (Identique à ce qu'on a utilisé pendant l'entraînement !)
    # Pour l'inférence, on resize l'image par exemple en 256x256 (ou 512) pour éviter 
    # les problèmes de mémoire, car le U-Net attend des images dont les dimensions 
    # sont des multiples de 16.
    target_size = 256
    
    transform = transforms.Compose([
        transforms.Resize((target_size, target_size)),
        transforms.ToTensor() # Transforme l'image [0, 255] PIL en Tensor [0, 1] C,H,W
    ])
    
    input_tensor = transform(pil_image).unsqueeze(0).to(device) # Ajout de la dimension Batch (1, 3, 256, 256)
    
    print(f"Image chargée. Shape du tenseur d'entrée: {input_tensor.shape}")
    
    # Affichage de l'image de test
    plt.figure(figsize=(6, 6))
    plt.imshow(input_tensor[0].cpu().permute(1, 2, 0).numpy())
    plt.title("Image en entrée du modèle U-Net")
    plt.axis('off')
    plt.show()
else:
    print(f"⚠️ Image de test introuvable: {test_image_path}")

## 3. Prédiction du Motion Flow
On fait passer l'image dans le réseau pour récupérer le Flow X et Y.

In [ ]:
if 'input_tensor' in locals():
    with torch.no_grad(): # On ne veut pas calculer les gradients (économise de la VRAM)
        pred_flow = model(input_tensor)
        
    # pred_flow est de shape [1, 2, H, W]
    # On extrait le flow_x et flow_y (on repasse sur CPU et en Numpy pour l'affichage)
    flow_x = pred_flow[0, 0, :, :].cpu().numpy()
    flow_y = pred_flow[0, 1, :, :].cpu().numpy()

    print(f"Flow généré ! Shape: {flow_x.shape}. Modèle est confiant sur des valeurs entre [{flow_x.min():.4f}, {flow_x.max():.4f}]")

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(flow_x, cmap='coolwarm')
    plt.title("Motion Flow Prédit: Déplacement en X")
    plt.colorbar()

    plt.subplot(1, 2, 2)
    plt.imshow(flow_y, cmap='coolwarm')
    plt.title("Motion Flow Prédit: Déplacement en Y")
    plt.colorbar()
    plt.show()

## 4. Application du Flow sur l'image (Warping)
Comme dans le premier notebook, on réutilise le vecteur produit par l'IA pour tordre l'image d'origine.

In [ ]:
if 'input_tensor' in locals():
    # On récupère l'image sous format numpy pour Scipy
    img_np = input_tensor[0].cpu().permute(1, 2, 0).numpy()
    h, w, c = img_np.shape
    
    # Grille de base
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    
    # IMPORTANT : Si le modèle a été entraîné avec une fonction Loss qui 
    # pénalisait à zéro, ou s'il n'est pas très bien entraîné, 
    # on peut "multiplier" l'amplitude du flow prédit pour forcer à voir l'effet localement (-1.0 à 1.0).
    # Vous pouvez changer ce multiplicateur.
    flow_multiplier = 10.0 
    
    m_y = y - (flow_y * flow_multiplier)
    m_x = x - (flow_x * flow_multiplier)
    
    first_warp = np.zeros_like(img_np)
    for channel in range(c):
        first_warp[..., channel] = map_coordinates(
            img_np[..., channel], 
            [m_y, m_x], 
            order=1,
            mode='reflect'
        )

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(img_np)
    plt.title("Avant (Statique)")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(first_warp)
    plt.title("Après (Warped par le Flow Prédit)")
    plt.axis('off')
    plt.show()